In [ ]:
# PEFT and quantized models eval generation

In [ ]:
! pip install datasets transformers[torch] 'accelerate>=0.26.0' peft bitsandbytes

In [ ]:
from datasets import load_dataset

def load_io_dataset(split, uf_n=90):
    dataset = load_dataset("json", data_files=f"TTE_uf{uf_n}/TTE_with_IO_{split}.json")

    def format_example(example):
        return {
            "text": f"Prompt: {example['input']}\n\n{example['output']}\n"
        }

    dataset = dataset["train"].map(format_example)
    return dataset

train_dataset = load_io_dataset("train")
test_dataset = load_io_dataset("test")
eval_dataset = load_io_dataset("eval")
print(train_dataset)
print(test_dataset)
print(eval_dataset)

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import AutoPeftModelForCausalLM

model_name = "../Qwen/models--Qwen--Qwen3-4B/snapshots/1cfa9a7208912126459214e8b04321603b3df60c"

inference_model = AutoPeftModelForCausalLM.from_pretrained(
    "./90_2048_fine_tuned_adapters_qwen3-4b",
    device_map="auto",
    #torch_dtype="auto"
)

tokenizer = AutoTokenizer.from_pretrained(model_name)

In [ ]:
import torch
from tqdm.auto import tqdm   # optional, for nicer progress
import pandas as pd

# Parallel LLM generation of evaluation's results
def perform_and_return_comparison_batched(batch):
    # batch is a dict with lists: {'input': [...], 'text': [...]}
    inputs = tokenizer(
        batch['input'],
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=2048,  # ← set sensible value, e.g. 2048
        padding_side='left'
    ).to("cuda")

    with torch.inference_mode():           # saves a bit of memory
        outputs = inference_model.generate(
            **inputs,
            max_new_tokens=600,
            #min_new_tokens=100,
        )

    decoded = tokenizer.batch_decode(
        outputs,
        skip_special_tokens=True
    )

    wo = pd.DataFrame()
    wo['dec']=decoded
    wo.to_json(f"batched_tmp/partial.json")
    return {
        #'results': results,
        'decoded': decoded}   # list of dicts — datasets will flatten it

# This is usually enough parallelism for one GPU
eval_output = eval_dataset.map(
    perform_and_return_comparison_batched,
    batched=True,
    batch_size=64,          # ← tune this: 4–16 most common sweet spot
                           # larger = better GPU util, but risk of OOM
    desc="Evaluating",
    remove_columns=eval_dataset.column_names,  # optional — keep only new fields
)

eval_output.to_json("90_2048_4B_dec_eval_processed_parallel.json")

In [ ]:
eval_dataset = load_io_dataset("eval")

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import AutoPeftModelForCausalLM

model_name = "../Qwen3-1.7B"

inference_model = AutoPeftModelForCausalLM.from_pretrained(
    "./90_2048_fine_tuned_adapters_qwen3-1.7b",
    device_map="auto",
    #torch_dtype="auto"
)

tokenizer = AutoTokenizer.from_pretrained(model_name)

In [ ]:
import torch
from tqdm.auto import tqdm   # optional, for nicer progress
import pandas as pd
# Parallel LLM generation of evaluation's results

def perform_and_return_comparison_batched(batch):
    # batch is a dict with lists: {'input': [...], 'text': [...]}
    inputs = tokenizer(
        batch['input'],
        return_tensors="pt",padding=True,
        truncation=True,
        max_length=2048,  # ← set sensible value, e.g. 2048
        padding_side='left'
    ).to("cuda")

    with torch.inference_mode():           # saves a bit of memory
        outputs = inference_model.generate(
            **inputs,
            max_new_tokens=600
        )

    decoded = tokenizer.batch_decode(
        outputs,
        skip_special_tokens=True
    )

    wo = pd.DataFrame()
    wo['dec']=decoded
    wo.to_json(f"batched_tmp/partial.json")
    return {
        #'results': results,
        'decoded': decoded}   # list of dicts — datasets will flatten it

# This is usually enough parallelism for one GPU
eval_output = eval_dataset.map(
    perform_and_return_comparison_batched,
    batched=True,
    batch_size=64,          # ← tune this: 4–16 most common sweet spot
                           # larger = better GPU util, but risk of OOM
    desc="Evaluating",
    remove_columns=eval_dataset.column_names,  # optional — keep only new fields
)

eval_output.to_json("90_2048_1.7B_dec_eval_processed_parallel.json")

In [ ]:
eval_dataset = load_io_dataset("eval")

In [ ]:
model_name = "./LA-framework-quant/results/quantized_model/FT_Qwen3-4B_90_2048"

# load the tokenizer and the model

inference_model = AutoPeftModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    #torch_dtype=torch.bfloat16
)

model_name = "../Qwen/models--Qwen--Qwen3-4B/snapshots/1cfa9a7208912126459214e8b04321603b3df60c"

tokenizer = AutoTokenizer.from_pretrained(model_name)


In [ ]:
import torch
from tqdm.auto import tqdm   # optional, for nicer progress
import pandas as pd

# Parallel LLM generation of evaluation's results

def perform_and_return_comparison_batched(batch):
    # batch is a dict with lists: {'input': [...], 'text': [...]}
    inputs = tokenizer(
        batch['input'],
        return_tensors="pt",padding=True,
        truncation=True,
        max_length=2048,  # ← set sensible value, e.g. 2048
        padding_side='left'
    ).to("cuda")

    with torch.inference_mode():           # saves a bit of memory
        outputs = inference_model.generate(
            **inputs,
            max_new_tokens=600
        )

    decoded = tokenizer.batch_decode(
        outputs,
        skip_special_tokens=True
    )

    wo = pd.DataFrame()
    wo['dec']=decoded
    wo.to_json(f"batched_tmp/partial.json")
    return {
        #'results': results, 
        'decoded':decoded
    }   # list of dicts — datasets will flatten it


# This is usually enough parallelism for one GPU
eval_output = eval_dataset.map(
    perform_and_return_comparison_batched,
    batched=True,
    batch_size=64,          # ← tune this: 4–16 most common sweet spot
                           # larger = better GPU util, but risk of OOM
    desc="Evaluating",
    remove_columns=eval_dataset.column_names,  # optional — keep only new fields
)

eval_output.to_json("90_4B_peft_quantized_dec_eval_processed_parallel.json")

In [ ]:
eval_dataset = load_io_dataset("eval")

In [ ]:
model_name = "./LA-framework-quant/results/quantized_model/FT_Qwen3-1.7B_90_2048"

# load the tokenizer and the model

inference_model = AutoPeftModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    #torch_dtype=torch.bfloat16
)

model_name = "../Qwen3-1.7B"

tokenizer = AutoTokenizer.from_pretrained(model_name)

In [ ]:
import torch
from tqdm.auto import tqdm   # optional, for nicer progress
import pandas as pd

# Parallel LLM generation of evaluation's results

def perform_and_return_comparison_batched(batch):
    # batch is a dict with lists: {'input': [...], 'text': [...]}
    inputs = tokenizer(
        batch['input'],
        return_tensors="pt",padding=True,
        truncation=True,
        max_length=2048,  # ← set sensible value, e.g. 2048
        padding_side='left'
    ).to("cuda")

    with torch.inference_mode():           # saves a bit of memory
        outputs = inference_model.generate(
            **inputs,
            max_new_tokens=600
        )

    decoded = tokenizer.batch_decode(
        outputs,
        skip_special_tokens=True
    )

    wo = pd.DataFrame()
    wo['dec']=decoded
    wo.to_json(f"batched_tmp/partial.json")
    return {
        #'results': results, 
        'decoded':decoded
    }   # list of dicts — datasets will flatten it


# This is usually enough parallelism for one GPU
eval_output = eval_dataset.map(
    perform_and_return_comparison_batched,
    batched=True,
    batch_size=64,          # ← tune this: 4–16 most common sweet spot
                           # larger = better GPU util, but risk of OOM
    desc="Evaluating",
    remove_columns=eval_dataset.column_names,  # optional — keep only new fields
)

eval_output.to_json("90_1.7B_peft_quantized_dec_eval_processed_parallel.json")